# 8.6 Exercise 2 – SVM for Spam Classification

Use the **real UCI SMS Spam Collection** dataset to classify messages as spam or ham using word-frequency features and message length.

The `ucimlrepo` import method can fail because the UCI repository currently lists this dataset but its standardized Python import is not available. This notebook therefore downloads the original UCI dataset file directly.


In [ ]:
import pandas as pd
import numpy as np
import requests
import zipfile
import io

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip"

response = requests.get(url, timeout=30)
response.raise_for_status()

with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    print(z.namelist())
    with z.open("SMSSpamCollection") as f:
        df = pd.read_csv(
            f,
            sep="\\t",
            header=None,
            names=["label", "message"],
            encoding="latin-1"
        )

print("Dataset shape:", df.shape)
df.head()


In [ ]:
df["label"] = df["label"].map({"ham": 0, "spam": 1})
df["message_length"] = df["message"].str.len()

print(df["label"].value_counts())
print(df[["message", "message_length", "label"]].head())


### Create word-frequency features

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack

vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    max_features=3000
)

X_words = vectorizer.fit_transform(df["message"])
X_length = df[["message_length"]].values

X = hstack([X_words, X_length])
y = df["label"].values

print("Final feature matrix shape:", X.shape)


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = LinearSVC()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Ham", "Spam"]))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


### Predict new messages

In [ ]:
new_messages = [
    "Congratulations! You have won a free prize. Call now!",
    "Are we meeting for lunch today?"
]

new_words = vectorizer.transform(new_messages)
new_lengths = np.array([[len(msg)] for msg in new_messages])
new_X = hstack([new_words, new_lengths])

predictions = model.predict(new_X)

for message, prediction in zip(new_messages, predictions):
    print("Message:", message)
    print("Prediction:", "Spam" if prediction == 1 else "Ham")
    print()


### Result

The real **UCI SMS Spam Collection** is used. The model uses word-frequency/TF-IDF features together with message length and applies an SVM classifier to predict **Spam** or **Ham**.
